In [0]:
# Import required libraries
from pyspark.sql.functions import *

Read log-data recursively from directories

In [0]:
df_logs_raw = spark.read.format('text').option("recursiveFileLookup", "true").load('/Volumes/workspace/default/logs/06Feb')

Extract required values from the read log-lines for further analysis

In [0]:
df_logs_enriched = (
    df_logs_raw.withColumns({
        "ts" : to_timestamp(substring(col('value'),0,19)),
        'items' : split(col('value'), '-'),
        "script_name" : col('items')[3],
        "operation_intance_id" : concat(col('items')[4],lit("-"),col('items')[5],lit("-"),col('items')[6],lit("-"),col('items')[7],lit("-"),col('items')[8]),
        'has_api_call': expr("value ilike '%API call to%'")

    }
    )
).drop('items').filter("has_api_call = 'true'")
    

Generated required insights from the data by grouping based on 'script_name' and counting the executions

In [0]:
df_logs_enriched.filter('ts >= "2026-03-04T19:30:00" and ts <= "2026-03-05T00:30:00"')\
    .groupBy('script_name')\
    .count()\
    .orderBy('count', ascending=False)\
    .withColumn('script_name', replace(concat(replace(split(col('script_name'),lit("__"))[0], lit(".py"),lit("")),lit(".py")   ), lit(" .py"),lit(".py")))\
    .display()